In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

import statsmodels.api as sm
from statsmodels.tsa.filters.hp_filter import hpfilter
from scipy.stats import norm
from scipy import stats


In [3]:
ruta = "/Users/bautistagianfrancisco/Library/Mobile Documents/com~apple~CloudDocs/MIA 2026/2 Trim/Modelado Estocastico/Bases de Datos MIA103/Ejemplo_Casa.xls"

casa = pd.read_excel(
    ruta,
    sheet_name="HPRICE"
)

casa.head()

,PRECIO,LOTE,CUARTOS,BANOS,PISOS,ENTRADA,REC,SOTANO,CALEF,AIRE,GARAGE,NBHD
0,42000,5850,3,1,2,1,0,1,0,0,1,0
1,38500,4000,2,1,1,1,0,0,0,0,0,0
2,49500,3060,3,1,1,1,0,0,0,0,0,0
3,60500,6650,3,1,2,1,1,0,0,0,0,0
4,61000,6360,2,1,1,1,0,0,0,0,0,0


In [5]:
variables = ['LOTE', 'CUARTOS', 'BANOS', 'PISOS', 'ENTRADA', 'REC',
       'SOTANO', 'CALEF', 'AIRE', 'GARAGE', 'NBHD']
X = sm.add_constant(casa[variables])
y = casa['PRECIO']
regmul = sm.OLS(y, X).fit()
print(regmul.summary())

                            OLS Regression Results                            
Dep. Variable:                 PRECIO   R-squared:                       0.673
Model:                            OLS   Adj. R-squared:                  0.666
Method:                 Least Squares   F-statistic:                     99.97
Date:                Wed, 26 Aug 2026   Prob (F-statistic):          6.18e-122
Time:                        21:52:28   Log-Likelihood:                -6034.1
No. Observations:                 546   AIC:                         1.209e+04
Df Residuals:                     534   BIC:                         1.214e+04
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -4038.3504   3409.471     -1.184      0.2

Manteniendo constantes las demás variables del modelo, una casa con garage tiene
un precio esperado aproximadamente 4244 dólares mayor que una casa sin garage.

Como garage es una variable dummy, el coeficiente se interpreta como una
diferencia promedio entre el grupo con garage y el grupo sin garage, controlando
por el resto de las variables incluidas en la regresión.

B)

El estadístico F sirve para testear la significatividad global de la regresión.

Es decir, permite evaluar si las variables explicativas incluidas en el modelo,
tomadas en conjunto, tienen capacidad para explicar la variabilidad del precio
de las casas.

La hipótesis nula es que todos los coeficientes de pendiente son iguales a cero:

H0: beta_LOTE = beta_CUARTOS = beta_BANOS = beta_PISOS = beta_ENTRADA
    = beta_REC = beta_SOTANO = beta_CALEF = beta_AIRE = beta_GARAGE
    = beta_NBHD = 0

La hipótesis alternativa es que al menos uno de esos coeficientes es distinto
de cero:

HA: al menos un beta_j != 0

Bajo la hipótesis nula, el modelo restringido sería:

PRECIO = alpha + u

Es decir, un modelo con intercepto solamente, sin variables explicativas.

Bajo la hipótesis alternativa, el modelo no restringido es la regresión múltiple:

PRECIO = alpha + beta1*LOTE + beta2*CUARTOS + beta3*BANOS + beta4*PISOS
       + beta5*ENTRADA + beta6*REC + beta7*SOTANO + beta8*CALEF
       + beta9*AIRE + beta10*GARAGE + beta11*NBHD + u

El estadístico F compara cuánto mejora el ajuste al pasar del modelo restringido
al modelo no restringido. Se calcula comparando la suma de cuadrados explicada
por la regresión contra la suma de cuadrados residual, ajustando por los grados
de libertad:

F = (SSR / k) / (SSE / (n - k - 1))

donde k es la cantidad de variables explicativas, n es la cantidad de
observaciones, SSR es la suma de cuadrados explicada por el modelo y SSE es la
suma de cuadrados de los residuos.

En la salida, el estadístico F es grande y su p-value es prácticamente 0. Como
el p-value es menor al 5%, rechazo la hipótesis nula. Por lo tanto, las variables
explicativas son conjuntamente significativas para explicar el precio de las
casas.

In [6]:
print("F-statistic:", regmul.fvalue)
print("p-value del F:", regmul.f_pvalue)

F-statistic: 99.96773763049414
p-value del F: 6.177730808645855e-122


In [7]:
regmul.f_pvalue < 0.05

np.True_

Puede haber alguna sospecha de multicolinealidad si las variables explicativas
están fuertemente relacionadas entre sí. Por ejemplo, casas más grandes pueden
tener más cuartos, más baños, más pisos, garage, aire, etc. Entonces varias
variables pueden estar capturando dimensiones parecidas de calidad/tamaño de la
vivienda.

Sin embargo, en esta regresión no aparece una señal fuerte de multicolinealidad
problemática. Una señal típica sería tener un R² alto y un F global significativo,
pero muchos coeficientes individualmente no significativos por errores estándar
inflados. En la Regresión 1 ocurre lo contrario: el F global es significativo y
la mayoría de los coeficientes también son individualmente significativos.

La única variable relativamente débil es CUARTOS, con un p-value cercano a 0.07,
pero eso por sí solo no alcanza para afirmar que haya un problema severo de
multicolinealidad.



In [10]:
casa_d = pd.DataFrame({
    "const": [1],
    "LOTE": [5100],
    "CUARTOS": [3],
    "BANOS": [2],
    "PISOS": [2],
    "ENTRADA": [1],
    "REC": [1],
    "SOTANO": [0],
    "CALEF": [1],
    "AIRE": [0],
    "GARAGE": [0],
    "NBHD": [1],
})

precio_esperado = regmul.predict(casa_d)

precio_esperado

0    94728.795701
dtype: float64

# Ejercicio 2

In [15]:
casa_reg = casa[casa["BANOS"] != 4].copy()

casa_reg["DB2"] = (casa_reg["BANOS"] == 2).astype(int)
casa_reg["DB3"] = (casa_reg["BANOS"] == 3).astype(int)

In [16]:
variables_reg2 = [
    "LOTE", "CUARTOS", "PISOS", "ENTRADA", "REC",
    "SOTANO", "CALEF", "AIRE", "GARAGE", "NBHD",
    "DB2", "DB3"
]

X2 = sm.add_constant(casa_reg[variables_reg2])
y2 = casa_reg["PRECIO"]

regmul2 = sm.OLS(y2, X2).fit()

print(regmul2.summary())

                            OLS Regression Results                            
Dep. Variable:                 PRECIO   R-squared:                       0.665
Model:                            OLS   Adj. R-squared:                  0.657
Method:                 Least Squares   F-statistic:                     88.01
Date:                Wed, 26 Aug 2026   Prob (F-statistic):          7.04e-118
Time:                        23:04:08   Log-Likelihood:                -6022.1
No. Observations:                 545   AIC:                         1.207e+04
Df Residuals:                     532   BIC:                         1.213e+04
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.031e+04   3464.573      2.976      0.0

El coeficiente de DB3 es aproximadamente 29392.54. Como DB3 es una dummy que vale
1 si la casa tiene exactamente 3 baños y 0 en caso contrario, y como la categoría
base omitida son las casas con 1 baño, este coeficiente mide la diferencia
esperada de precio entre una casa con 3 baños y una casa comparable con 1 baño.

Manteniendo constantes el lote, cuartos, pisos, entrada, cuarto de recreación,
sótano, calefacción, aire, garage y barrio, una casa con 3 baños tiene un precio
esperado aproximadamente 29392.54 dólares canadienses mayor que una casa con 1
baño.

In [18]:
beta_lote = regmul2.params["LOTE"]
se_lote = regmul2.bse["LOTE"]

li = beta_lote - 1.96 * se_lote
ls = beta_lote + 1.96 * se_lote

li, ls

(np.float64(2.839735917296012), np.float64(4.213770771130296))

Para obtener el intervalo de confianza del 95% para el coeficiente de LOTE uso:

IC 95% = beta_hat ± 1.96 * SE(beta_hat)

De la Regresión 2, el coeficiente estimado de LOTE es aproximadamente 3.5268 y
su error estándar es aproximadamente 0.3505.

Entonces:

IC 95% = 3.5268 ± 1.96 * 0.3505

IC 95% ≈ [2.84 ; 4.21]

Esto significa que, con un 95% de confianza, el efecto marginal de un pie
cuadrado adicional de lote sobre el precio esperado de la casa se encuentra
aproximadamente entre 2.84 y 4.21 dólares canadienses, manteniendo constantes las
demás variables del modelo.

In [20]:
test_g = regmul2.t_test("DB3 = 2 * DB2")

print(test_g)

                             Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0          2430.1762   5704.059      0.426      0.670   -8775.066    1.36e+04


Se desea testear si el coeficiente de DB3 es igual al doble del coeficiente de
DB2:

H0: beta_DB3 = 2 beta_DB2

HA: beta_DB3 != 2 beta_DB2

Esta hipótesis puede escribirse como una restricción lineal:

H0: beta_DB3 - 2 beta_DB2 = 0

La intuición del test es evaluar si el efecto esperado de tener 3 baños, medido
respecto de la categoría base de 1 baño, es exactamente el doble del efecto
esperado de tener 2 baños respecto de esa misma categoría base.

En la regresión, beta_DB2 es aproximadamente 13481.18 y beta_DB3 es
aproximadamente 29392.54. La diferencia beta_DB3 - 2 beta_DB2 es
aproximadamente 2430.18.

Sin embargo, el test de restricción lineal arroja un estadístico t cercano a
0.426 y un p-value aproximado de 0.670. Como el p-value es mayor que 0.05, no
rechazo la hipótesis nula al 5%.

Por lo tanto, con esta muestra no hay evidencia estadística suficiente para
afirmar que el efecto de tener 3 baños sea distinto del doble del efecto de tener
2 baños.